<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# NYC Bike Counts + Location + Streets

### CP255: Urban Informatics and Visualization


[NYC DOT Bicycle Counts](https://data.cityofnewyork.us/Transportation/Bicycle-Counts/uczf-rk3c) · [NYC DOT Bicycle Counters](https://data.cityofnewyork.us/Transportation/Bicycle-Counters/smn3-rzf9) · [NYC DCP LION street network](https://www.nyc.gov/content/planning/pages/resources/datasets/lion) · [`sjoin_nearest()` reference](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Bike Counts + Location + Streets

<hr style="border: 1px solid #fdb515;" />

We have two datasets, bike counts over time and the locations of the counters.
They come in separate files.

* The counts file is large.
  Each row is the number of bikes one counter recorded in one 15-minute interval.
* The counters file is small.
  Each row is one counter with its latitude and longitude, which we will turn into point geometry.

Geometry makes a table slower to work with and harder to read, so we keep it on the small table and only attach it when we need a map.

The plan for this session

1. Clean the timestamps in the counts table and summarise it to one row per station (total counts, number of readings, first and last date).
2. Merge that summary with the counter locations and turn the result into a `GeoDataFrame`.
3. Bring in the Manhattan street network and find the nearest street segment to each station.

**Which street segment is each station closest to, and which streets have the most stations near them?**

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

<hr style="border: 1px solid #fdb515;" />

By the end of this session, you will be able to

1. Load bike counts (a time series) and counter locations (a table with latitude and longitude)
2. Convert longitude and latitude into point geometry and create a `GeoDataFrame`
3. Find and remove invalid coordinates such as `(0, 0)` with a boolean filter
4. Convert a string timestamp column into a pandas datetime column
5. Summarise counts per station with `groupby().agg()` (sum, count, min, max)
6. Merge the station summary with station geometry only when it is needed
7. Reproject to EPSG:2263 (US survey feet) before measuring distance
8. Use `sjoin_nearest()` to find the nearest street segment to each station and keep the distance
9. Count stations per street segment

### Data for this notebook

The data lives in `data/raw/` inside this week's folder.
It's already there, no download needed.

| File | What it holds |
|---|---|
| `Bicycle_Counts_2025_2023.parquet` | Recorded bicycle counts over time, one row per 15-minute interval per counter, January 2023 through February 2025 |
| `Bicycle_Counters_20250301.csv` | The bicycle counters, including their location |
| `lion_manh.parquet` | NYC LION street network, Manhattan only, in EPSG:2263 |

`data/raw/` is a read-only reference copy, so please don't edit or overwrite files there.
If you need to modify the data, load it into a variable (as below) and work with that copy instead.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Setup

<hr style="border: 1px solid #fdb515;" />



In [ ]:
!pip install "geopandas[all]"

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

# Course palette, used for every map in this notebook.
BERKELEY_BLUE = "#003262"
BERKELEY_GOLD = "#fdb515"

# Files this notebook writes go here, never into data/raw/.
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## A- Bicycle Count Data Analysis

<hr style="border: 1px solid #fdb515;" />

### **Dataset Description**

You will work with two datasets.
1. `Bicycle_Counts_2025_2023.parquet` has the recorded bicycle counts over time.
2. `Bicycle_Counters_20250301.csv` has the counters, including their location.

These are the fields we will use, and you will find them in one or both files.
- `id` identifies the counter location.
- `countid` identifies each recorded count, one per time interval.
- `date` is the timestamp of the count, stored as a string.
- `counts` is the number of bicycles counted in that interval.

#### Before any analysis, open `Bicycle_Counters_20250301.csv` from the left hand menu, then look at the counts table once you load it below. Which one does not have any geographic data?

### **A - Read and sanity check our data**

In [ ]:
# Load bicycle count data - simple tabular data, stored as parquet to keep the file small

bike = pd.read_parquet('data/raw/Bicycle_Counts_2025_2023.parquet')

Similar to the `bike` dataframe, we create a `bike_counters_df` DataFrame by loading the dataset `Bicycle_Counters_20250301.csv`

In [ ]:
# Load bicycle counter location data
bike_counters_df = pd.read_csv('data/raw/Bicycle_Counters_20250301.csv')

hmmmmmmm but we now read bicycle counters data with `pd.read_csv()`, and it is still not a geodataframe.

Do you remember what we did last week to convert the *longitude, latitude* columns to `geopandas` geospatial data?

In [ ]:
# construct shapely point geometry using gpd.point_from_xy()
# remember: pass longitude to x and latitude to y

counters_geom = gpd.points_from_xy(
    x=bike_counters_df['longitude'],
    y=bike_counters_df['latitude'],
)

# now, we can use geopandas Geodataframe constructor to create a new geodataframe
# remember geodataframes are dataframes + geometry column

counters_gdf = gpd.GeoDataFrame(
    bike_counters_df,
    geometry=counters_geom,
    crs=4326,
)

Lets see if we have the correct locations

In [ ]:
counters_gdf.plot()

hmmmmmmm this looks off ... what is happening?😯 looks like we have some 0,0 locations??
Ok, let's go to our sanity checks and inspect the data types, and value range.

In [ ]:
counters_gdf.describe()

Found it!! The minimum of my `latitude` column is 0.
The minimum of `longitude` won’t help me much here, because longitude values are supposed to be negative in the U.S., but its maximum is 0, which is just as suspicious.
But the plot is telling us something more specific. This looks like a **(0, 0)** point (both latitude and longitude are zero).

this is definitely a data entry error.
Let's see how many problematic points do we have?

In [ ]:
# hint: check for both latitude AND longitude being zero
is_zero_zero = (
    (counters_gdf["latitude"] == 0.0) &
    (counters_gdf["longitude"] == 0.0)
)

# tip: to know how many rows meet your condition, do a sum().
# python treats False as 0 and True as 1, so the sum shows you the total number of True you've got
is_zero_zero.sum()

So, only 1 entry was messing with the whole thing!😒

Let's remove that and check again!

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 1 - Write code to remove that row and plot again.

You should decide how you want to remove it.
Think about what that filter gives you.

🔔 Resist the temptation to ask a GenAI.
Try to think first, and if you don’t remember, try playing around with what you think might be close.
Linger a bit with that uncomfortable feeling.
You’re building the instincts that make you faster later.

In [ ]:
""" # BEGIN PROMPT
counters_clean_gdf = ...

# Plot here

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
# Option 1: keep only the "good" rows (often the clearest)
counters_clean_gdf = counters_gdf.loc[~is_zero_zero]

# Option 2: drop by index (nice if you want to inspect the problematic rows first)
# counters_clean_gdf = counters_gdf.drop(index=counters_gdf.index[is_zero_zero])

# Plot here
counters_clean_gdf.plot()
# END SOLUTION

In [ ]:
assert isinstance(counters_clean_gdf, gpd.GeoDataFrame), "Keep the result a GeoDataFrame."
assert len(counters_clean_gdf) == len(counters_gdf) - 1, "Exactly one row should be removed."
assert ((counters_clean_gdf["latitude"] == 0) & (counters_clean_gdf["longitude"] == 0)).sum() == 0, "The (0, 0) row is still there."
assert len(counters_gdf) == 40, "Filter into a new variable; leave counters_gdf as loaded." 

Here are two hints for you

- Option 1 is to keep only the "good" rows (often the clearest), with `.loc[]` and the negated filter `~is_zero_zero`.
- Option 2 is to drop by index (nice if you want to inspect the problematic rows first), with `.drop()`.

### Sanity check the data

1. Display the first 5 rows of each dataset.

2. Print a summary of each dataset.

In [ ]:
# Now, lets take a look at column names: looks good here
counters_clean_gdf.info()

In [ ]:
# column names look good here too
bike.info()

#### Note that we have a column with the same name `id` in both datasets. Is this useful or redundant?

It's useful. `id` is the shared key that connects the counts over time (in `bike`) to the counter locations (in `bike_counters_df`).

Looking at the column types, we have a date column that can be quite useful as we've seen.
But it has the wrong type, lets fix that.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Converting `date` to datetime

<hr style="border: 1px solid #fdb515;" />

Right now `bike["date"]` is a string.
We convert it to a datetime type so we can sort by time, filter by date, and find the first and last timestamp.

In this dataset the date strings look like `2023-06-03 00:00:00`, which is year-month-day hour:minute:second.

### Convert this format in pandas

```python
bike["dt_fmt"] = pd.to_datetime(
    bike["date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
```

#### Format specifiers
| Specifier | Meaning |
|-----------|---------|
| `%Y`      | Four-digit year (e.g., `2023`) |
| `%m`      | Month (zero-padded, e.g., `06`) |
| `%d`      | Day (zero-padded, e.g., `03`) |
| `%H`      | Hour (24-hour format, zero-padded, e.g., `14`) |
| `%M`      | Minute (zero-padded, e.g., `00`) |
| `%S`      | Second (zero-padded, e.g., `00`) |

Two more you will meet in other datasets. `%I` is the hour in 12-hour format and `%p` is the AM/PM marker.

#### Alternative - Convert without explicit format
If all values follow the same structure, `pd.to_datetime()` can infer the format.
```python
df["date_fmt"] = pd.to_datetime(df['date_col_name']) #replace 'date_col_name' with the name of the column that has date in it.
```
This works, but passing `format=` is faster on large tables, and it fails loudly when a value does not match.

To learn more about python date formatting, [read this documentation.](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes)

Once the column is a datetime, the `.dt` accessor gives you the `date`, `time`, `year`, `month` and `day` for the whole column.

`df["date_formatted"].dt.year`  # gets the year for all the rows in `df["date_formatted"]`.

### After conversion - split into date and time

```python
bike["date_fmt"] = bike["dt_fmt"].dt.date
bike["time_fmt"] = bike["dt_fmt"].dt.time
```

### Check the time range

```python
bike["dt_fmt"].min(), bike["dt_fmt"].max()
```

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 2 - Convert Date Column to Datetime Format

The `date` column is a string in the format `year-month-day hour:minute:second`.

Write your code to
1. Convert the `date` column to datetime and save it as a new column called `dt_fmt`.
   The format string is `"%Y-%m-%d %H:%M:%S"`.
2. Extract the date and the time into two new columns, `date_fmt` and `time_fmt`.
3. Print the minimum and maximum timestamps in the dataset.

In [ ]:
""" # BEGIN PROMPT
## Your code here

bike["dt_fmt"] = ...

bike['date_fmt'] = ...                             # Extract date
bike['time_fmt'] = ...                             # Extract time

# verify the range

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
## Your code here

bike["dt_fmt"] = pd.to_datetime(
    bike["date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

bike['date_fmt'] = bike["dt_fmt"].dt.date          # Extract date
bike['time_fmt'] = bike["dt_fmt"].dt.time          # Extract time

# verify the range
bike["dt_fmt"].min(), bike["dt_fmt"].max()
# END SOLUTION

In [ ]:
assert pd.api.types.is_datetime64_any_dtype(bike["dt_fmt"]), "dt_fmt should be a datetime column."
assert bike["dt_fmt"].isna().sum() == 0, "Every date string matches the format, so no NaT is expected."
assert bike["dt_fmt"].min() == pd.Timestamp("2023-01-01 00:00:00"), "Check the format string."
assert "date_fmt" in bike.columns and "time_fmt" in bike.columns, "Add date_fmt and time_fmt with the .dt accessor." 

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 3 - Exploring Unique Identifiers

1. How many unique `countid` values exist?
2. Is every `countid` unique in the dataset?

`.nunique()` and `.shape` will help.

Store the number in `n_unique_countid` and the yes/no answer in `all_unique`.

In [ ]:
""" # BEGIN PROMPT
## Your code here: How many unique `countid` values exist?
n_unique_countid = ...

## Your code here: Is every `countid` unique in the dataset?
all_unique = ...

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
## Your code here: How many unique `countid` values exist?
n_unique_countid = bike["countid"].nunique()

## Your code here: Is every `countid` unique in the dataset?
all_unique = n_unique_countid == bike.shape[0]

print(n_unique_countid, all_unique)
# END SOLUTION

In [ ]:
assert n_unique_countid == bike["countid"].nunique(), "Use .nunique() on the countid column."
assert bool(all_unique) is True, "Compare the number of unique values with the number of rows." 

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Aggregating Bicycle Counts by Location (`id`)

<hr style="border: 1px solid #fdb515;" />

Right now your table has many rows per location, one per time interval.
We want a new table with one row per location `id`.

To do that we take two steps.

1. Group by `id`, so each location becomes a group.
2. Aggregate each group into summary values such as sum, count, min and max.

For each `id`, compute

1. `total_count`, the total bikes counted across all intervals (sum)
2. `num_intervals`, how many readings we have for this `id` (count of rows)
3. `min_date`, the earliest timestamp (min)
4. `max_date`, the latest timestamp (max)

#### How `.agg()` works

Inside `.agg()` you define each new column with the pattern

`new_column_name = (which_column_to_use, which_summary_function)`

For example, `total_count = ("counts", "sum")`.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 4 - Build the station summary

1. Group by `id` and compute the four metrics with `.agg()`.
2. Store the result in a new DataFrame called `bike_by_id`.
3. Show the top 10 locations by `total_count`.

Type the code below rather than pasting it, and make sure you understand each part as you type.

```python
bike_by_id = (
    bike.groupby("id", as_index=False)
        .agg(
            total_count=("counts", "sum"),
            num_intervals=("counts", "count"),
            min_date=("dt_fmt", "min"),
            max_date=("dt_fmt", "max"),
        )
)

# top 10 locations by total bicycles counted
bike_by_id.sort_values(
  "total_count",
  ascending=False
  ).head(10)
```

In [ ]:
""" # BEGIN PROMPT
# type the above code here
bike_by_id = ...

# view the first 10 rows

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
# type the above code here
bike_by_id = (
    bike.groupby("id", as_index=False)
        .agg(
            total_count=("counts", "sum"),
            num_intervals=("counts", "count"),
            min_date=("dt_fmt", "min"),
            max_date=("dt_fmt", "max"),
        )
)

# view the first 10 rows
bike_by_id.sort_values(
  "total_count",
  ascending=False
  ).head(10)
# END SOLUTION

In [ ]:
assert set(["id", "total_count", "num_intervals", "min_date", "max_date"]).issubset(bike_by_id.columns), "Name the four metrics as in the code above."
assert len(bike_by_id) == bike["id"].nunique(), "One row per station id."
assert bike_by_id["total_count"].sum() == bike["counts"].sum(), "total_count should sum the counts column."
assert bike_by_id["num_intervals"].sum() == len(bike), "num_intervals should count rows per id." 

### What happened?

You collapsed a long time-series table, many rows per station, into a station summary, one row per `id`.

* `bike.groupby("id", as_index=False)` split the table into one group per station and kept `id` as a regular column rather than the index.
* `.agg(...)` computed the summary columns for each group.

  * `total_count` is the sum of `counts` for that station.
  * `num_intervals` is how many readings we have for that station.
  * `min_date` and `max_date` are the earliest and latest timestamps.

So `bike_by_id` is a compact summary table, one row per station.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 5 - Busiest Locations

The total count alone is not always the best metric, because stations have different numbers of readings.

- Compute the average count per interval for each location, using `total_count` and `num_intervals`.
- Sort by the highest average count.
- Display the top 10 locations.

In [ ]:
""" # BEGIN PROMPT
## Your code here
## hint: you can divide the entire column by another and pandas will do it cell-wise for you
bike_by_id['avg_counts'] = ...

## Sort - use ascending with bool to control the behavior
bike_by_id_sorted = ...

## Show the top 10 locations

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
## Your code here
## hint: you can divide the entire column by another and pandas will do it cell-wise for you
bike_by_id['avg_counts'] = bike_by_id["total_count"] / bike_by_id["num_intervals"]

## Sort - use ascending with bool to control the behavior
bike_by_id_sorted = bike_by_id.sort_values("avg_counts", ascending=False)

## Show the top 10 locations
bike_by_id_sorted.head(10)
# END SOLUTION

In [ ]:
assert "avg_counts" in bike_by_id.columns, "Add avg_counts to bike_by_id."
assert (bike_by_id["avg_counts"] - bike_by_id["total_count"] / bike_by_id["num_intervals"]).abs().max() < 1e-9, "avg_counts = total_count / num_intervals."
assert bike_by_id_sorted["avg_counts"].is_monotonic_decreasing, "Sort with the highest average first." 

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Merge counts with counters (so we can do geospatial analysis)

<hr style="border: 1px solid #fdb515;" />

Up to now we have been working with two separate tables.

* `bike_by_id` tells us what happened over time, but not where.
* `counters_clean_gdf` tells us where each counter is, but has no counts.

To do geospatial analysis we need both in the same table.
That is why the `id` column exists.
It appears in both datasets, so we can match rows on it.

Both tables have one row per `id`, so we merge on `id`.
After the merge every station row has its counter's geometry, and the result can become a `GeoDataFrame` that we can map, spatially join, and analyse by neighborhood or proximity.

#### ❗❗❗Why did we have them separated anyways?

What is the justification behind having two separate tables?
Think, reflect, talk with your pairs and write the answer below.

*Your answer here*

In [ ]:
# Keep only the geometry + id (and maybe lat/lon if you want them)
counter_cols = ["id", "geometry"]
counters_loc = counters_clean_gdf[counter_cols].copy()

### Lets do a left merge here

We merge the station summary `bike_by_id` with the locations table `counters_loc` on the shared key `id`.
We use a left merge, `how="left"`, because we want to keep every row from `bike_by_id` even if a few `id`s are missing from the counters file.

* If an `id` matches, the row gets its `geometry` and we can map it.
* If it does not match, `geometry` is `NaN`.
  That is useful, because it flags stations where we have counts but no location.

An inner merge would drop those rows silently, which is harder to debug.

In [ ]:
# Merge station-level summary with station locations
bike_by_id_geo = bike_by_id.merge(
    counters_loc,
    on="id",
    how="left",
)

# how many station rows found no location?
bike_by_id_geo["geometry"].isna().sum()

In [ ]:
#check the type of the resulting object
type(bike_by_id_geo)

After the merge, `bike_by_id_geo` is still a regular pandas DataFrame.
It contains a `geometry` column, but pandas does not treat that column as anything special.

For geospatial operations such as mapping, spatial joins, buffering and distance, we need a `GeoDataFrame`.
That means telling GeoPandas that `geometry` is the active geometry column.

```python
bike_by_id_geo = bike_by_id_geo.set_geometry("geometry")
```

Once you run `set_geometry("geometry")`, methods like `.plot()` and `.sjoin()` work the way you expect.
The geometry column keeps its CRS from `counters_loc`, so the result is in EPSG:4326 without saying so again.

**Tip** - If that fails for any reason, the more explicit way is to re-create the GeoDataFrame.

```python
bike_by_id_geo = gpd.GeoDataFrame(bike_by_id_geo, geometry="geometry", crs=4326)
```

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 6 - Set the active geometry

Type the code you see above, and set the active geometry.
Then check the type again.

In [ ]:
""" # BEGIN PROMPT
bike_by_id_geo = ...

# check the type again

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
bike_by_id_geo = bike_by_id_geo.set_geometry("geometry")

# check the type again
print(type(bike_by_id_geo))
print(bike_by_id_geo.crs)
# END SOLUTION

In [ ]:
assert isinstance(bike_by_id_geo, gpd.GeoDataFrame), "Set the active geometry column."
assert bike_by_id_geo.crs is not None and bike_by_id_geo.crs.to_epsg() == 4326, "The CRS should carry over as EPSG:4326."
assert len(bike_by_id_geo) == len(bike_by_id), "Keep every station row." 

#### Now Lets Visualize!!

In [ ]:
bike_by_id_geo.explore(
    column="total_count",
    scheme="EqualInterval",
    k=5,
    marker_type="circle_marker",
    marker_kwds={"radius": 6, "fill": True},
    legend=True,
    legend_kwds={"caption": "Total counts (binned)"},
    tooltip=["id", "total_count"],
    cmap="Blues",
    style_kwds={"color": BERKELEY_BLUE, "weight": 1, "fillOpacity": 0.85},
    tiles="Esri.WorldGrayCanvas",
)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## B. Add Street Networks

<hr style="border: 1px solid #fdb515;" />

### Street networks + bike stations (why we are doing this)

We want to answer one question.

**When a bike counter reports a number, which street segment should get credit for that count?**

Right now the counters are points.
Planning decisions are made on streets, so we will snap each counter to its nearest street segment in `data/raw/lion_manh.parquet`, then count how many counters land on each segment.

Distance is only meaningful in a projected CRS, in feet or meters rather than degrees.
For NYC we use EPSG:2263, US survey feet.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 7 - Read, reproject, plot NYC LION street network data

1. Read the NYC Street Network data `data/raw/lion_manh.parquet` into a GeoDataFrame named `manh_st`
2. Reproject it to EPSG:2263 and store the result as `manh_st_2263`
3. Generate a simple plot to visualize the street network.

In [ ]:
""" # BEGIN PROMPT
# Your code here

manh_st = ...

manh_st_2263 = ...

# plot here

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
# Your code here

manh_st = gpd.read_parquet('data/raw/lion_manh.parquet')

manh_st_2263 = manh_st.to_crs(2263)

# plot here
manh_st_2263.plot(
    color="gray",
    linewidth=0.3,
    figsize=(6, 9),
)
# END SOLUTION

In [ ]:
assert isinstance(manh_st, gpd.GeoDataFrame), "Read the parquet file with gpd.read_parquet()."
assert manh_st_2263.crs.to_epsg() == 2263, "Reproject to EPSG:2263."
assert len(manh_st_2263) == 35542, "Read the whole file; no filtering yet." 

LION already ships in EPSG:2263 (check `manh_st.crs`), so `.to_crs(2263)` changes nothing here.
We still call it so the code stays correct if a future extract arrives in a different CRS.

In [ ]:
# .info() wont show the details if you have too many columns (can you find how many is too many?)
# answer: pandas' display.max_info_columns, 100 by default; LION has 129
# lets check the column names

manh_st_2263.columns.tolist()

Too many columns, lets only keep a subset that can be useful.

In [ ]:
## Creat a list of columns we want to keep
columns_to_keep = ["SegmentID", "PhysicalID", "GenericID", "Street",
                   "SAFStreetName", "FeatureTyp", "SegmentTyp", "TrafDir",
                   "BikeLane", "BIKE_TRAFDIR", "POSTED_SPEED", "StreetWidth_Min",
                   "StreetWidth_Max", "Number_Total_Lanes", "SHAPE_Length",
                   "NodeIDFrom", "NodeIDTo", "geometry"]

LION is a centerline file for more than streets.
Its `FeatureTyp` code separates public streets (`"0"`, which includes bridges and greenway bike paths) from subway and rail lines (`"1"`), shorelines (`"2"`), census and district boundaries (`"3"`, `"7"`, `"8"`), ferry routes (`"F"`), and a few other feature types.
A station snapped to a census tract boundary would be nonsense, so we keep code `"0"` only.

In [ ]:
## Your code here
# Use that list to keep only keep selected column, and only public street segments (FeatureTyp "0")

is_street = manh_st_2263["FeatureTyp"] == "0"
manh_st_filtered = manh_st_2263.loc[is_street, columns_to_keep]

## check the columns
# manh_st_filtered.columns
print(len(manh_st_filtered), "street segments")

### Nearest neighbors with stations (points) and streets (lines)

We have two layers.

* `bike_by_id_geo`, the bike stations, as points
* `manh_st_filtered`, the street segments, as LineStrings

**For each station, find the closest street segment and store the distance.**

That attaches each station to one street segment, so we can count stations per segment later.

<hr style="border: 1px solid #fdb515;" />

### ✏️ Try It Out 8 - Same CRS on both sides

Lets make sure our station data is in the same crs.
Reproject `bike_by_id_geo` to EPSG:2263 and store it as `bike_by_id_geo_2263`.

In [ ]:
""" # BEGIN PROMPT
# lets make sure our station data is in the same crs
bike_by_id_geo_2263 = ...

"""; # END PROMPT
# BEGIN SOLUTION NO PROMPT
# lets make sure our station data is in the same crs
bike_by_id_geo_2263 = bike_by_id_geo.to_crs(2263)

print(bike_by_id_geo_2263.crs == manh_st_filtered.crs)
# END SOLUTION

In [ ]:
assert bike_by_id_geo_2263.crs.to_epsg() == 2263, "Reproject to EPSG:2263."
assert bike_by_id_geo.crs.to_epsg() == 4326, "Reproject into a new variable; leave bike_by_id_geo in EPSG:4326." 

#### Nearest street for each station (and the distance)

In [ ]:
stations_nearest_street = bike_by_id_geo_2263.sjoin_nearest(
    manh_st_filtered,
    max_distance=1600,   # about 0.3 miles, a 6 minute walk
    how="inner",
    distance_col="dist_ft",
)

stations_nearest_street

Now every station row has:

* the street segment it matched to (via `index_right` and whatever street attributes exist)
* `dist_ft`: distance between the station point and that street segment (in feet)

Two things to notice:

* Only 15 of the 31 stations appear.
  The other 16 are in Brooklyn, Queens, the Bronx, or Staten Island, more than 1,600 ft from any Manhattan street, so the inner join dropped them.
* There are 18 rows, not 15.
  The three Willis Avenue Bridge stations are exactly 1,480 ft from two LION rows that share one `SegmentID`, so each one matched twice.

#### Which street segments have the most stations near them?

After `sjoin_nearest`, every station row has an `index_right` value, the street segment it matched to.
The code below does two things.

1. Group by `index_right`, which puts all stations that matched the same segment together.
2. Count how many stations are in each group, and store that as `num_stations`.

Then sort by `num_stations` to see which segments have the most stations.

In [ ]:
stations_per_street = (
    stations_nearest_street.groupby("index_right", as_index=False)
    .size()
    .rename(columns={"size": "num_stations"})
)

# Top 10 street segments by number of stations attached
stations_per_street.sort_values("num_stations", ascending=False).head(10)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ⭐ Optional - Visualize the connectors, the lines joining each station to the nearest point on its street

<hr style="border: 1px solid #fdb515;" />


In [ ]:
from shapely import shortest_line

In [ ]:
# Generate LineStrings that connect each station with its nearest street segment
lines = []
for _, row in stations_nearest_street.iterrows():
    street_geom = manh_st_filtered.loc[row["index_right"], "geometry"]
    line = shortest_line(row["geometry"], street_geom)
    lines.append(line)

# Create a new GeoDataFrame out of these connector lines
connectors = gpd.GeoDataFrame({"geometry": lines}, crs=manh_st_filtered.crs)

# Length of each connector line = distance between station and nearest street
connectors["dist_ft"] = connectors.length

connectors

In [ ]:
# zoom in to see the gold connectors
m = manh_st_filtered[["geometry"]].explore(
    color="gray",
    style_kwds={"weight": 1, "opacity": 0.5},
    tiles="Esri.WorldGrayCanvas",
)
m = stations_nearest_street.explore(
    m=m,
    color=BERKELEY_BLUE,
    tooltip=["id", "Street", "dist_ft"],
)
m = connectors.explore(
    m=m,
    color=BERKELEY_GOLD,
    style_kwds={"weight": 3},
)
m

In [ ]:
m.save(OUTPUTS / "stations_nearest_street.html")

Notes

* `sjoin_nearest()` works across geometry types, so points and lines together is fine.
* Distance comes from the geometries, so project first (EPSG:2263 here).
* If two streets are equally near, a station can match twice.
  That is normal, so decide how to handle ties before you count.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary - Street-Level Attachment of Raw Counts

<hr style="border: 1px solid #fdb515;" />

| Step | Takeaway |
|---|---|
| Keep counts and locations in separate tables | Keep expensive geometry operations on the small locations table, not the large time-series table |
| Boolean filter on `(latitude == 0) & (longitude == 0)` | A quick `.describe()` and plot can surface data-entry errors before any analysis runs |
| `pd.to_datetime()` on the `date` column | Unlocks time-based sorting, filtering, and min/max range checks |
| `groupby("id").agg(...)` | Collapses a per-interval time series into one row per station |
| `.merge(..., how="left")` then `.set_geometry()` | Attach geometry only when it is time to map, and keep every station row |
| `.to_crs(2263)` before measuring | Feet, not degrees |
| `sjoin_nearest(max_distance=, distance_col=)` | Snap points to the nearest line and keep the distance; watch for ties and for points beyond the cutoff |
| `groupby("index_right").size()` | Count how many stations attached to each segment |